# Day 8 — Files, JSON, CSV, context managers
Objectives:
- Read/write text and CSV/JSON.
- Use `with` for resource safety.
- Robust error handling.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-08`. Read
`python/ds-60day/companion-guides/day08_files_json_csv_context.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

A file handle is a stateful resource with a current cursor and an open
or closed lifetime. A context manager (`with`) makes ownership visible:
acquire the resource, use it inside the block, and release it on both
success and failure. Text files also have an encoding and newline
policy; make both explicit at boundaries.

JSON and CSV are representations, not automatically trusted schemas.
JSON preserves nested object/list structure but has a small type system.
CSV is a table of text fields and requires column-name and conversion
policy. Parse and validate rows near the boundary, keep writes
recoverable, and report expected read/decode failures without hiding
programming errors.

### Vocabulary

- **file handle:** an open resource used to read or write a file.
- **cursor:** the current read/write position in a stream.
- **context manager:** an object that performs setup and guaranteed cleanup around a `with` block.
- **encoding:** the mapping between text characters and bytes.
- **serialization:** converting in-memory data to a storable representation.
- **schema:** the expected fields, types, and constraints of data.

## Syntax anatomy

`with path.open("r", encoding="utf-8") as handle:` creates a context,
binds the open handle, and guarantees `close` at block exit.
`json.load(handle)` reads JSON from a file object; `json.loads(text)`
parses an already-loaded string. For CSV output, `newline=""` lets the
`csv` module manage platform newline rules correctly.

### Worked example 1 — Round-trip JSON through an in-memory stream

Observe serialization without creating learner files. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
import io
import json

record = {"name": "Ada", "skills": ["Python", "SQL"], "active": True}
stream = io.StringIO()
json.dump(record, stream, ensure_ascii=False)
encoded = stream.getvalue()
decoded = json.loads(encoded)
(encoded, decoded == record)

**Expected observation:** The JSON text and `True` are displayed. Serialization creates text; parsing reconstructs equivalent Python data.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Read CSV rows as dictionaries

Column headers become keys while every field initially remains text. Predict first; then run the next cell.

In [ ]:
import csv
import io

csv_text = "name,score\nAda,9\nLin,10\n"
rows = list(csv.DictReader(io.StringIO(csv_text)))
(rows, type(rows[0]["score"]).__name__)

**Expected observation:** `([{'name': 'Ada', 'score': '9'}, {'name': 'Lin', 'score': '10'}], 'str')`. Numeric-looking CSV fields still require conversion.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Use `repr` on a short read and inspect the cursor with `handle.tell()` when content seems missing.
2. Catch `FileNotFoundError`, `PermissionError`, and `json.JSONDecodeError` separately when the recovery differs.
3. Open CSV with `newline=''` and text files with an explicit `encoding='utf-8'`.
4. Write to a temporary sibling and replace the destination only after a successful serialization.

**Alternative to compare:** Use `json.loads`/`dumps` for in-memory text and `load`/`dump` for open file objects; use a database or columnar format when CSV cannot express the needed contract.

**Boundary to test:** Missing files, malformed records, absent or extra CSV columns, non-ASCII names, partial writes, and empty files need deliberate behavior.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
import csv
import json
from pathlib import Path

artifact_dir = Path('artifacts/day08')
artifact_dir.mkdir(parents=True, exist_ok=True)
p = artifact_dir / 'people.json'
people = [{"name": "Ada", "age": 36}, {"name": "Alan", "age": 41}]
with p.open('w', encoding='utf-8') as f:
    json.dump(people, f, indent=2)

with p.open(encoding='utf-8') as f:
    loaded = json.load(f)
loaded

# CSV
p2 = artifact_dir / 'people.csv'
with p2.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['name', 'age'])
    writer.writeheader()
    writer.writerows(people)
p2.read_text(encoding='utf-8').splitlines()[:3]


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Implement `safe_load_json(path)` that returns parsed data on success and `None` only for a missing file or malformed JSON, logging which expected failure occurred. **Constraints:** open with UTF-8, catch `FileNotFoundError` and `json.JSONDecodeError` explicitly, and let unrelated errors surface.
   **Verify:** test a valid file, a missing path, and an invalid JSON file inside a temporary directory.

2. Implement `csv_to_records(path)` and `records_to_csv(records, path)` using `csv.DictReader` and `csv.DictWriter`. **Contract:** headers define keys, output field order is explicit, and numeric conversion policy is documented. **Constraints:** use UTF-8 and `newline=''`; do not hand-split comma-delimited text.
   **Verify:** round-trip two records including a non-ASCII name and compare the normalized records.

### Additional mastery practice

Treat files as fallible boundaries. Specify encoding, newline behavior, schema, and failure policy, and keep writes recoverable.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

3. **Prediction:** After `handle.read(2)` on a text file containing `abcd`, predict what a second `handle.read()` returns and explain the cursor.
   **Progressive hint:** Reads advance the file object's current position.
   **Verify:** Assert the first read is `'ab'`, the second is `'cd'`, and a third is empty; record cursor positions `2` and `4`.
4. **Tracing:** Trace when a file is open and closed through a `with` block, including when JSON decoding raises inside the block.
   **Progressive hint:** Context-manager cleanup runs on both normal and exceptional exit.
   **Verify:** Record `handle.closed` inside and after both successful and failing `with` blocks; assert it is true after either exit.
5. **Implementation:** Implement an atomic JSON writer that writes a sibling temporary file then replaces the destination.
   **Progressive hint:** Use explicit UTF-8, `json.dump`, and `Path.replace`.
   **Verify:** Read the replaced destination and assert it contains the complete new JSON; simulate serialization failure and assert the old destination is still intact.
6. **Debugging:** Repair CSV writing that creates blank lines on Windows or corrupts non-ASCII names.
   **Progressive hint:** Open with `newline=''` and `encoding='utf-8'`.
   **Verify:** Round-trip two CSV rows including a non-ASCII name on the current platform; assert no blank records and exact decoded characters.
7. **Edge case and explanation:** Design a CSV reader policy for missing columns and extra columns; return accepted rows and quarantined row/error pairs.
   **Progressive hint:** Validate each row at the boundary rather than failing much later.
   **Verify:** Use one valid, one missing-column, and one extra-column row; assert accepted plus quarantined equals input and each quarantine reason names its violated rule.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Implement `safe_load_json(path)` that returns parsed data on success and `None` only for a missing file or malformed JSON, logging which expected failure occurred. **Constraints:** open with UTF-8, catch `FileNotFoundError` and `json.JSONDecodeError` explicitly, and let unrelated errors surface. **Verify:** test a valid file, a missing path, and an invalid JSON file inside a temporary directory.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Implement `safe_load_json(path)` that returns parsed data on success and `None` only for a missing file or malformed JSON, logging which expected failure occurred. open with UTF...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Implement `csv_to_records(path)` and `records_to_csv(records, path)` using `csv.DictReader` and `csv.DictWriter`. **Contract:** headers define keys, output field order is explicit, and numeric conversion policy is documented. **Constraints:** use UTF-8 and `newline=''`; do not hand-split comma-delimited text. **Verify:** round-trip two records including a non-ASCII name and compare the normalized records.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Implement `csv_to_records(path)` and `records_to_csv(records, path)` using `csv.DictReader` and `csv.DictWriter`. headers define keys, output field order is explicit, and numeri...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** After `handle.read(2)` on a text file containing `abcd`, predict what a second `handle.read()` returns and explain the cursor. **Progressive hint:** Reads advance the file object's current position. **Verify:** Assert the first read is `'ab'`, the second is `'cd'`, and a third is empty; record cursor positions `2` and `4`.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: After `handle.read(2)` on a text file containing `abcd`, predict what a second `handle.read()` returns and explain the cursor. Reads advance the file object's current position....
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace when a file is open and closed through a `with` block, including when JSON decoding raises inside the block. **Progressive hint:** Context-manager cleanup runs on both normal and exceptional exit. **Verify:** Record `handle.closed` inside and after both successful and failing `with` blocks; assert it is true after either exit.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Trace when a file is open and closed through a `with` block, including when JSON decoding raises inside the block. Context-manager cleanup runs on both normal and exceptional ex...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Implement an atomic JSON writer that writes a sibling temporary file then replaces the destination. **Progressive hint:** Use explicit UTF-8, `json.dump`, and `Path.replace`. **Verify:** Read the replaced destination and assert it contains the complete new JSON; simulate serialization failure and assert the old destination is still intact.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Implement an atomic JSON writer that writes a sibling temporary file then replaces the destination. Use explicit UTF-8, `json.dump`, and `Path.replace`. Read the replaced destin...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair CSV writing that creates blank lines on Windows or corrupts non-ASCII names. **Progressive hint:** Open with `newline=''` and `encoding='utf-8'`. **Verify:** Round-trip two CSV rows including a non-ASCII name on the current platform; assert no blank records and exact decoded characters.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Repair CSV writing that creates blank lines on Windows or corrupts non-ASCII names. Open with `newline=''` and `encoding='utf-8'`. Round-trip two CSV rows including a non-ASCII...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Design a CSV reader policy for missing columns and extra columns; return accepted rows and quarantined row/error pairs. **Progressive hint:** Validate each row at the boundary rather than failing much later. **Verify:** Use one valid, one missing-column, and one extra-column row; assert accepted plus quarantined equals input and each quarantine reason names its violated rule.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Design a CSV reader policy for missing columns and extra columns; return accepted rows and quarantined row/error pairs. Validate each row at the boundary rather than failing muc...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
